4. Does The Guardian respond to casualty spikes differently depending on region, and has any disparity narrowed or widened over time (2015–2024)?

In [40]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import pearsonr

In [41]:
monthly = pd.read_csv('..\data\master_monthly.csv')
master  = pd.read_csv('..\data\master.csv')
wb      = pd.read_csv('..\data\worldbank_income_clean.csv')
 
wb_region = wb[['country', 'region_group']].drop_duplicates()
 
# Join monthly
monthly = monthly.drop(columns='region_group')
monthly = monthly.merge(wb_region, on='country', how='inner')
 
# Join master
master = master.drop(columns='region_group')
master = master.merge(wb_region, on='country', how='inner')

df = monthly[monthly['monthly_fatalities'].notna()].copy()

yearly_agg = df.groupby(['year', 'region_group']).agg(
    total_articles   = ('article_count',     'sum'),
    total_fatalities = ('monthly_fatalities', 'sum')
).reset_index()

Helper function

In [42]:
def axis_ref(col):
    return "x domain" if col == 1 else f"x{col} domain"
 
def yaxis_ref(col):
    return "y domain" if col == 1 else f"y{col} domain"
 
 
def build_scatter(master_data, region, color, title):
    """
    Scatter plot of casualties vs articles for one region.
    Prints Pearson r when called.
    """
    sub = master_data[
        (master_data['region_group'] == region) &
        (master_data['total_fatalities'].notna()) &
        (master_data['total_fatalities'] > 0)
    ].copy()
 
    r_val, p_val = pearsonr(sub['total_fatalities'], sub['total_articles'])
    print(f"\n{region} country-level Pearson: r = {r_val:.4f}, p = {p_val:.4e}, n = {len(sub)}")
 
    fig = go.Figure()
 
    fig.add_trace(go.Scatter(
        x=sub['total_fatalities'], y=sub['total_articles'],
        mode='markers+text',
        marker=dict(color=color, size=8, opacity=0.7),
        text=sub['country'],
        textposition='top center',
        textfont=dict(size=8),
        hovertemplate=(
            '<b>%{text}</b><br>'
            'Casualties: %{x:,.0f}<br>'
            'Articles: %{y:,}<extra></extra>'
        )
    ))
 
    # OLS trend line
    z = np.polyfit(sub['total_fatalities'], sub['total_articles'], 1)
    x_line = np.linspace(sub['total_fatalities'].min(), sub['total_fatalities'].max(), 100)
    fig.add_trace(go.Scatter(
        x=x_line, y=np.polyval(z, x_line),
        mode='lines',
        line=dict(color=color, width=2, dash='dash'),
        showlegend=False
    ))
 
    fig.add_annotation(
        text=f"r = {r_val:.3f}, p = {p_val:.2e}, n = {len(sub)}",
        xref="x domain", yref="y domain",
        x=0.95, y=0.95,
        xanchor='right', yanchor='top',
        showarrow=False,
        font=dict(size=14),
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor=color, borderwidth=1
    )
 
    fig.update_layout(
        title=title,
        xaxis_title='Total Casualties',
        yaxis_title='Total Articles',
        template='plotly_white',
        width=1000, height=600,
        showlegend=False
    )
 
    return fig
 
 
def build_top10_bars(master_data, region, color_art, color_cas, title):
    """
    Two horizontal bar charts stacked vertically:
      Top: top 10 by article count
      Bottom: top 10 by casualty count
    """
    sub = master_data[
        (master_data['region_group'] == region) &
        (master_data['total_fatalities'].notna()) &
        (master_data['total_fatalities'] > 0)
    ]
 
    top_articles = sub.nlargest(10, 'total_articles').sort_values('total_articles')
    top_casualties = sub.nlargest(10, 'total_fatalities').sort_values('total_fatalities')
 
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=(
            f'{region}: Top 10 by Article Count',
            f'{region}: Top 10 by Casualty Count'
        ),
        vertical_spacing=0.15
    )
 
    fig.add_trace(go.Bar(
        y=top_articles['country'],
        x=top_articles['total_articles'],
        orientation='h',
        marker_color=color_art,
        text=top_articles['total_articles'].apply(lambda x: f'{x:,}'),
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>Articles: %{x:,}<extra></extra>'
    ), row=1, col=1)
 
    fig.add_trace(go.Bar(
        y=top_casualties['country'],
        x=top_casualties['total_fatalities'],
        orientation='h',
        marker_color=color_cas,
        text=top_casualties['total_fatalities'].apply(lambda x: f'{x:,.0f}'),
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>Casualties: %{x:,.0f}<extra></extra>'
    ), row=2, col=1)
    
    art_max = top_articles['total_articles'].max()
    cas_max = top_casualties['total_fatalities'].max()
    
    fig.update_xaxes(title_text='Total Articles', range=[0, art_max * 1.25], row=1, col=1)
    fig.update_xaxes(title_text='Total Casualties', range=[0, cas_max * 1.25], row=2, col=1)
 
    fig.update_layout(
        title=title,
        template='plotly_white',
        width=900, height=700,
        showlegend=False
    )
 
    return fig
 
 
def build_pearson_by_year(monthly_data):
    """
    Line chart of Pearson r by year for both regions.
    Prints correlation stats when called.
    """
    df = monthly_data[monthly_data['monthly_fatalities'].notna()].copy()
 
    yearly = df.groupby(['country', 'year', 'region_group']).agg(
        total_articles   = ('article_count',     'sum'),
        total_fatalities = ('monthly_fatalities', 'sum')
    ).reset_index()
 
    results = []
    for region in ['Global South', 'Global North']:
        for year in sorted(yearly['year'].unique()):
            sub = yearly[(yearly['region_group'] == region) & (yearly['year'] == year)]
            if len(sub) > 2:
                r_val, p_val = pearsonr(sub['total_fatalities'], sub['total_articles'])
                countries = sorted(sub['country'].tolist())
                results.append({
                    'year': year,
                    'region_group': region,
                    'r': r_val,
                    'p': p_val,
                    'n': len(sub),
                    'significant': p_val < 0.05,
                    'countries': countries
                })
 
    results_df = pd.DataFrame(results)
 
    print(f"\n{'='*60}")
    print("PEARSON r BY YEAR")
    print(f"{'='*60}")
    for _, row in results_df.iterrows():
        sig = '***' if row['p'] < 0.001 else '**' if row['p'] < 0.01 else '*' if row['p'] < 0.05 else ''
        print(f"  {row['region_group']:15s} {row['year']}  r={row['r']:.3f}  p={row['p']:.4e}  n={row['n']:>3} {sig}")
 
    gs = results_df[results_df['region_group'] == 'Global South']
    gn = results_df[results_df['region_group'] == 'Global North']
 
    fig = go.Figure()
 
    for subset, name, color in [(gs, 'Global South', '#E07A5F'), (gn, 'Global North', '#3D405B')]:
        hover_texts = []
        for _, row in subset.iterrows():
            c_list = row['countries']
            if len(c_list) > 15:
                c_str = ', '.join(c_list[:15]) + f'... (+{len(c_list)-15} more)'
            else:
                c_str = ', '.join(c_list)
            hover_texts.append(
                f"<b>{name} {row['year']}</b><br>"
                f"r = {row['r']:.3f}, p = {row['p']:.2e}<br>"
                f"n = {row['n']} countries<br>"
                f"<br>{c_str}"
            )
 
        fig.add_trace(go.Scatter(
            x=subset['year'], y=subset['r'],
            name=name,
            mode='lines+markers+text',
            line=dict(color=color, width=2),
            marker=dict(
                size=10,
                symbol=subset['significant'].apply(
                    lambda s: 'circle' if s else 'circle-open'
                ),
                color=color
            ),
            text=subset['n'].apply(lambda n: f'n={n}'),
            textposition='top center',
            textfont=dict(size=9, color=color),
            hovertemplate='%{customdata}<extra></extra>',
            customdata=hover_texts
        ))
 
    fig.add_hline(y=0, line_dash='dot', line_color='grey', opacity=0.5)
 
    fig.update_layout(
        title='Pearson Correlation (Casualties vs Articles) by Year and Region',
        xaxis_title='Year',
        yaxis_title='Pearson r',
        margin=dict(b=80),
        yaxis=dict(range=[-0.3, 1.0]),
        xaxis=dict(dtick=1),
        template='plotly_white',
        width=1000, height=550,
        legend=dict(x=0.01, y=0.01, xanchor='left', yanchor='bottom',
                    bgcolor='rgba(255,255,255,0.8)'),
        annotations=[dict(
            text='● filled = significant (p < 0.05)   ○ open = not significant',
            xref='paper', yref='paper',
            x=0.5, y=-0.18, showarrow=False,
            font=dict(size=11, color='grey')
        )]
    )
 
    return fig

def build_yearly_scatter(yearly, title):
    """
    Side-by-side scatter: Global South | Global North
    One point per year, with Pearson r.
    Prints stats when called.
    """
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Global South', 'Global North'),
        horizontal_spacing=0.12
    )

    print(f"\n{'='*60}")
    print(f"PEARSON – {title}")
    print(f"{'='*60}")

    for col, (region, color) in enumerate([
        ('Global South', '#E07A5F'),
        ('Global North', '#3D405B')
    ], start=1):
        sub = yearly[yearly['region_group'] == region]
        if len(sub) < 3:
            continue

        r_val, p_val = pearsonr(sub['total_fatalities'], sub['total_articles'])
        print(f"  {region}:  r = {r_val:.4f}, p = {p_val:.4e}, n = {len(sub)}")

        fig.add_trace(go.Scatter(
            x=sub['total_fatalities'], y=sub['total_articles'],
            mode='markers+text',
            marker=dict(color=color, size=10),
            text=sub['year'].astype(str),
            textposition='top center',
            textfont=dict(size=9),
            hovertemplate=(
                'Year: %{text}<br>'
                'Casualties: %{x:,.0f}<br>'
                'Articles: %{y:,}<extra></extra>'
            )
        ), row=1, col=col)

        z = np.polyfit(sub['total_fatalities'], sub['total_articles'], 1)
        x_line = np.linspace(sub['total_fatalities'].min(),
                             sub['total_fatalities'].max(), 100)
        fig.add_trace(go.Scatter(
            x=x_line, y=np.polyval(z, x_line),
            mode='lines',
            line=dict(color=color, width=2, dash='dash'),
            showlegend=False
        ), row=1, col=col)

        fig.add_annotation(
            text=f"r = {r_val:.3f}, p = {p_val:.2e}",
            xref=axis_ref(col), yref=yaxis_ref(col),
            x=0.95, y=0.95,
            xanchor='right', yanchor='top',
            showarrow=False,
            font=dict(size=13),
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor=color, borderwidth=1
        )

    fig.update_xaxes(title_text='Yearly Casualties', row=1, col=1)
    fig.update_xaxes(title_text='Yearly Casualties', row=1, col=2)
    fig.update_yaxes(title_text='Yearly Articles', row=1, col=1)

    fig.update_layout(
        title=title,
        template='plotly_white',
        width=1000, height=500,
        showlegend=False
    )

    return fig

In [43]:
fig_yearly_scatter = build_yearly_scatter(
    yearly_agg,
    'Yearly: Articles vs Casualties with Pearson r'
)
fig_yearly_scatter.show()


PEARSON – Yearly: Articles vs Casualties with Pearson r
  Global South:  r = 0.3910, p = 2.6384e-01, n = 10
  Global North:  r = 0.7856, p = 1.2096e-02, n = 9


We will investigate how responsive the Guardian is to casualty between Global North and Global South by finding the linear combination between yearly articles and casualties.

From the scatter graph above, we could see that these two variables have stronger correlation in Global North than in Global South, with the pearson correlation of 0.7856 compare to 0.3910. Also, consider the p-value of Global South which is 0.26 compared to 0.012 of Global North, we can conclude that it is more statistically significant that casualties have positive correlation to number of articles in Global North than in Global South



In [44]:
fig_gs_bars = build_top10_bars(
    master, 'Global South', '#E07A5F', '#F2CC8F',
    'Global South: Top 10 Countries by Coverage vs Casualties (2015–2024)'
)
fig_gs_bars.show()

fig_gs_scatter = build_scatter(
    master, 'Global South', '#E07A5F',
    'Global South: Total Casualties vs Total Articles by Country (2015–2024)'
)
fig_gs_scatter.show()


Global South country-level Pearson: r = 0.3302, p = 3.7789e-04, n = 112


As we already found out there was correlation between casualties and articles in Global South, we first find out whether top 10 countries with highest casualties were also top 10 countries with highest number of articles. The bar chart show that 5/10 countries appeared in both list

To find out whether the correlation still hold, we will also plot scatter graph between casualties and articles by country.

The above plot show there was statistically significant medium positive correlation of 0.330, indicates the Guardian does respond to casualties spike in this region

In [45]:
fig_gn_bars = build_top10_bars(
    master, 'Global North', '#3D405B', '#81B29A',
    'Global North: Top 10 Countries by Coverage vs Casualties (2015–2024)'
)
fig_gn_bars.show()

fig_gn_scatter = build_scatter(
    master, 'Global North', '#3D405B',
    'Global North: Total Casualties vs Total Articles by Country (2015–2024)'
)
fig_gn_scatter.show()


Global North country-level Pearson: r = 0.0946, p = 5.4116e-01, n = 44


When we do the same thing for the Global North, even though 6/10 countries appeared in both list, the number wasn't as coherent like in Global South analysis. The scatter graph proves that the correlation between casualties and articles in Global North was statistically insignificant and very weak. Hence, Global North related articles are driven by other reason such as geopolitical or reason related to the demographic of readers.

In [46]:
fig_pearson = build_pearson_by_year(monthly)
fig_pearson.show()
 


PEARSON r BY YEAR
  Global South    2015  r=0.462  p=5.5763e-04  n= 52 ***
  Global South    2016  r=0.529  p=8.3539e-06  n= 63 ***
  Global South    2017  r=0.619  p=3.0374e-08  n= 66 ***
  Global South    2018  r=0.321  p=8.7527e-04  n=104 ***
  Global South    2019  r=0.198  p=4.6067e-02  n=102 *
  Global South    2020  r=0.099  p=3.1274e-01  n=105 
  Global South    2021  r=0.529  p=5.2358e-10  n=120 ***
  Global South    2022  r=0.141  p=1.4886e-01  n=107 
  Global South    2023  r=0.149  p=1.1124e-01  n=115 
  Global South    2024  r=0.201  p=3.0641e-02  n=116 *
  Global North    2018  r=0.098  p=6.4824e-01  n= 24 
  Global North    2019  r=0.278  p=2.0961e-01  n= 22 
  Global North    2020  r=0.369  p=5.1318e-03  n= 56 **
  Global North    2021  r=0.239  p=5.1154e-02  n= 67 
  Global North    2022  r=0.415  p=8.8088e-04  n= 61 ***
  Global North    2023  r=0.222  p=8.8576e-02  n= 60 
  Global North    2024  r=0.133  p=3.1635e-01  n= 59 


To find out how responsive of the Guardian toward casualties changed overtime, we plot line graph show how pearson correlation changed by year. Noted that at some year, some countries have no casualties data due to ACLED limitation, so the number of countries included in each region each year is different

We can see that the Guardian is less responsive toward casualties spike in Global South over the year, with only exception of a spike in 2021. Meanwhile, in Global North, the Guardian responsiveness toward casualties spiked relatively unchanged, except in 2022, likely due to the start of Russia - Ukraine war

